In [3]:
import json, os, sys
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

In [22]:
system_prompt = """You are an expert in automated planning. Your task is to write a plan in pseudocode to aid a smaller LLM in completing its task. Do not solve its task yourself; only write the plan.

**Description of the small LLM:**
The small LLM is a web shopper that can navigate the web and find product offers. It can record information in its memory, read its memory, and interact with webpages. The small LLM must solve a small LLM task using these four webshops:

E-Store Athletes: http://localhost:8081/
TechTalk: http://localhost:8082/
CamelCases: http://localhost:8083/
Hardware Cafe: http://localhost:8084/

Then it must submit the final result. To submit the result, it must:
1. Navigate to the solution page: http://localhost:3000/
2. Enter the final result in the text field on the solution page. If there is no result to return after completion of the task, simply enter "Done" into the text field.
3. Press the "Submit Final Result" button.

**Your instructions:**
Write the high-level plan in Python code, using the following functions to call the small LLM. Don't overcomplicate the plan with special cases or granular details.

The small LLM can execute the following functions:
* search(store, product)-> string or None: Open the store and search for the product. Return the product page URL as a string, or None if not found. The small LLM will handle the search.
* open_page(url)->bool: Open the given URL in the small LLM's browser. Return True if successful, False otherwise.
* fill_text_field(field_description, text)->bool: The small LLM finds a text field matching the given description and enters the text. Return True if successful, False otherwise.
* press_button(button_description): Find a button on the current page and click it.
* prompt("your instructions"): If these functions are not sufficient, you can use the prompt function to give the small LLM instructions. The small LLM can only return unformatted text output.

Example: The small LLM task is to find the cheapest offer for Product P. Your output:
```
stores = [E-Store Athletes, TechTalk, CamelCases, Hardware Cafe]
results = []

for store in stores:
    url_or_none = search(store, P) # Return the product page URL or None if not found
    if url_or_none is not None:
        price = prompt("Extract the price from the product page.")
        results.append((url_or_none, price))

selected_url = min(results, key=lambda x: x[1])[0]

open_page("http://localhost:3000/")
fill_text_field("Solution field", selected_url)
press_button("Submit Final Result")
```
"""

task_prompt = """Small LLM task:
Find the cheapest price for the Asus ROG Ryujin II ARGB 360mm Liquid CPU Cooler.

Write the high-level plan in Python code, using the following functions to call the small LLM. Don't overcomplicate the plan with special cases or granular details.
"""

In [19]:
# test with gpt-5
result1 = client.chat.completions.create(
    model="gpt-5",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task_prompt},
    ],
)
print(result1.choices[0].message.content)

stores = [E-Store Athletes, TechTalk, CamelCases, Hardware Cafe]
product = "Asus ROG Ryujin II ARGB 360mm Liquid CPU Cooler"
offers = []

for store in stores:
    url_or_none = search(store, product)  # Return the product page URL or None if not found
    if url_or_none is not None:
        offers.append(url_or_none)

# Remove duplicates while preserving order
offers = list(dict.fromkeys(offers))

open_page("http://localhost:3000/")
if len(offers) == 0:
    fill_text_field("Solution field", "Done")
else:
    fill_text_field("Solution field", "\n".join(offers))
press_button("Submit Final Result")


In [23]:
# test with gpt-5-mini
result2 = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task_prompt},
    ],
)
print(result2.choices[0].message.content)

```
stores = ["E-Store Athletes", "TechTalk", "CamelCases", "Hardware Cafe"]
product = "Asus ROG Ryujin II ARGB 360mm Liquid CPU Cooler"
results = []

for store in stores:
    # Search the store for the product; returns product page URL or None
    product_url = search(store, product)
    if product_url is not None:
        # Open the product page so the agent can inspect it
        open_page(product_url)
        # Ask the agent to extract the price from the currently open product page.
        # Require a plain numeric value only (no currency symbols), e.g., 249.99.
        price_text = prompt(
            "On the currently open product page, extract the product price. "
            "Return only a numeric value representing the price in USD (e.g. 249.99). "
            "If no clear price is present, return NOT FOUND."
        )
        if price_text is not None and price_text != "NOT FOUND":
            # Convert the returned text to a number and record the result
            price = 